# OpenShorts on Kaggle (2×T4)

**Before running — notebook settings (right panel):**

| setting | value |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Internet | **On** (needs a phone-verified account) |

**Add-ons → Secrets** — create these and tick them for this notebook:

| secret | needed for |
|---|---|
| `GITHUB_TOKEN` | cloning the private repo (a PAT with `repo` scope) |
| `GEMINI_API_KEY` | clip selection + scene context — **without it nothing is clipped** |
| `ASSEMBLYAI_API_KEY` | transcription *with diarization*. Without it, faster-whisper runs instead and has no diarization — which the framing policy uses, so quality drops |
| `YOUTUBE_COOKIES` | paste the whole contents of a working `cookies.txt` |

Run the cells in order. Cell 4 tells you whether it actually works.

## 1 — Is the GPU actually there?

If this shows 0 GPUs, fix the accelerator setting before going further — everything below will still run, just ~25× slower.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}  devices={torch.cuda.device_count()}")

## 2 — Secrets and clone

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def load(name, required=False):
    try:
        os.environ[name] = secrets.get_secret(name)
        print(f"  ok       {name}")
        return True
    except Exception:
        print(f"  {'MISSING ' if required else 'not set '} {name}")
        return False

load("GEMINI_API_KEY", required=True)
load("ASSEMBLYAI_API_KEY")
load("YOUTUBE_COOKIES")
have_token = load("GITHUB_TOKEN")

BRANCH = "session/framing-work"
DEST = "/kaggle/working/openshorts"
url = (f"https://{os.environ['GITHUB_TOKEN']}@github.com/foskigr8/openshorts.git"
       if have_token else "https://github.com/foskigr8/openshorts.git")

if not os.path.isdir(DEST):
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, DEST],
                       capture_output=True, text=True)
    # Never print the URL on failure — it carries the token.
    print("clone ok" if r.returncode == 0 else f"CLONE FAILED: {r.stderr[-400:]}")
else:
    print("already cloned")

os.chdir(DEST)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 3 — Install and launch

First run takes **5–10 min** (pip + npm build). It ends by printing a public URL.

In [ ]:
!bash kaggle_bootstrap.sh

## 4 — Smoke test: is it *actually* working?

Starting is not the same as working. This checks the four things that
independently break, and says which one failed rather than just "error".

In [ ]:
import json, subprocess, urllib.request

def check(label, fn):
    try:
        ok, detail = fn()
    except Exception as e:
        ok, detail = False, f"{type(e).__name__}: {e}"
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}: {detail}")
    return ok

def api(path):
    with urllib.request.urlopen(f"http://localhost:8000{path}", timeout=15) as r:
        return r.status, r.read()

results = []

# 1. API alive
results.append(check("API", lambda: (api("/api/system")[0] == 200, "/api/system 200")))

# 2. Dashboard served from the SAME process (single-origin mode)
def _ui():
    status, body = api("/")
    return status == 200 and b"<div id=\"root\"" in body, f"/ returned {len(body)} bytes of HTML"
results.append(check("Dashboard", _ui))

# 3. GPU reachable from the app's own imports (not just nvidia-smi)
def _gpu():
    out = subprocess.run(["python3", "-c",
        "import torch;print(torch.cuda.is_available(), torch.cuda.device_count())"],
        capture_output=True, text=True).stdout.strip()
    return out.startswith("True"), out
results.append(check("CUDA in app env", _gpu))

# 4. NVENC present — without it every render silently falls back to x264 (slow)
def _nvenc():
    out = subprocess.run("ffmpeg -hide_banner -encoders 2>/dev/null | grep -c nvenc",
                         shell=True, capture_output=True, text=True).stdout.strip()
    return out.isdigit() and int(out) > 0, f"{out} nvenc encoders"
results.append(check("NVENC", _nvenc))

# 5. YouTube reachable with the cookie jar — FUNCTIONAL, not structural.
#    cookie_health.py only checks that cookie NAMES exist and will report OK
#    for a jar YouTube rejects, so it is deliberately not used here.
def _yt():
    r = subprocess.run(["yt-dlp", "--cookies", "cookies.txt", "--skip-download",
                        "--print", "%(title)s", "https://youtu.be/ua9Z0Lq3QVA"],
                       capture_output=True, text=True, timeout=120)
    title = (r.stdout or "").strip().splitlines()[-1] if r.stdout.strip() else ""
    return bool(title), title or (r.stderr or "").strip()[-200:]
results.append(check("YouTube + cookies", _yt))

print(f"\n{sum(results)}/{len(results)} passed")
if not all(results):
    print("\nIf 'YouTube + cookies' failed: the jar expired (~3h lifetime).")
    print("Re-paste YOUTUBE_COOKIES from a fresh local cookies.txt and re-run cell 2-3.")

## 5 — End-to-end render test (optional, ~1-2 min on GPU)

The real proof: reframe a short clip and check the framing telemetry.
Uses a source you upload or download yourself, so it works even if cookies
are dead.

In [ ]:
# Point SRC at any short landscape video with speech (upload one to
# /kaggle/working, or attach a Kaggle Dataset and use /kaggle/input/...).
SRC = "/kaggle/working/test.mp4"

import os, time
if not os.path.exists(SRC):
    print(f"put a video at {SRC} first (or edit SRC)")
else:
    os.environ.setdefault("USE_ASD", "1")
    import reframe_v2 as r
    t0 = time.time()
    # No transcript here, so diarization is unavailable and LR-ASD carries the
    # speaker identification alone — a deliberately harder case than production.
    r.render(SRC, "/kaggle/working/test_vertical.mp4", 0.75)
    print(f"\nrendered in {time.time()-t0:.1f}s -> /kaggle/working/test_vertical.mp4")
    print("Look at the '🎯 Framing evidence' line above:")
    print("  lip-sync high + size ~0%  = working as intended")
    print("  size high                 = the speaker signal is not reaching the camera")

## 6 — Keep the session alive

The bootstrap backgrounds everything, so without this the cell finishes and
Kaggle can reap the session while a render is still running.

**Outputs do not survive.** `/kaggle/working` is wiped when the session ends
(12h cap). Download clips through the dashboard before stopping, or set the
`AWS_*` secrets so `s3_uploader.py` pushes them to S3/B2 as they finish.

In [ ]:
import time
while True:
    time.sleep(60)
    print(".", end="", flush=True)